# SCRFD 얼굴 검출 v10: 고각도 CCTV 후보 우선

DB와 ArcFace 식별을 제외하고 SCRFD 얼굴 검출 성공에만 집중합니다. confidence가 충분한 후보는 `FACE`, 낮지만 얼굴 가능성이 있는 후보는 `REVIEW`로 표시합니다. 완전한 정수리·뒷머리는 SCRFD의 범위가 아니며 별도 head/person detector가 필요합니다.

In [1]:
from __future__ import annotations

import ctypes
import json
import os
import sys
import time
from pathlib import Path

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'deeplearning').is_dir() and (candidate / 'webapps').is_dir():
            return candidate
    raise RuntimeError('smart_office_monitoring 저장소 안에서 실행하세요.')

def load_env_file(path: Path) -> None:
    if not path.is_file():
        return
    for raw in path.read_text(encoding='utf-8-sig').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = (item.strip() for item in line.split('=', 1))
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {'\"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)

PROJECT_ROOT = find_project_root()
for env_path in (
    PROJECT_ROOT / 'deeplearning/training/.env.face',
    PROJECT_ROOT / 'deeplearning/training/.env.local',
    PROJECT_ROOT / 'deeplearning/training/.env',
):
    load_env_file(env_path)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_ROOT = PROJECT_ROOT / 'deeplearning/.models'
DETECTOR_PATH = Path(
    os.environ.get('FACE_DETECTION_MODEL_PATH')
    or MODEL_ROOT / 'scrfd/scrfd_10g_bnkps.onnx'
).resolve()
FACE_RTSP_URL = os.environ.get('FACE_RTSP_URL', '').strip()
CAMERA_INDEX = int(os.environ.get('CAMERA_INDEX', '0'))
INPUT_SIZE = int(os.environ.get('FACE_DETECTION_INPUT_SIZE', '1280'))
CANDIDATE_THRESHOLD = float(os.environ.get('FACE_CANDIDATE_THRESHOLD', '0.30'))
FACE_THRESHOLD = float(os.environ.get('FACE_CONFIRMED_THRESHOLD', '0.60'))
FULL_INTERVAL = int(os.environ.get('FACE_FULL_DETECTION_INTERVAL', '2'))
TILE_INTERVAL = int(os.environ.get('FACE_TILE_DETECTION_INTERVAL', '10'))
TILE_ROWS = int(os.environ.get('FACE_TILE_ROWS', '2'))
TILE_COLUMNS = int(os.environ.get('FACE_TILE_COLUMNS', '2'))
TILE_OVERLAP = float(os.environ.get('FACE_TILE_OVERLAP', '0.20'))
TRACK_STALE_CYCLES = int(os.environ.get('FACE_BOX_STALE_CYCLES', '3'))
DISPLAY_FULLSCREEN = os.environ.get('FACE_DISPLAY_FULLSCREEN', 'true').lower() in {
    '1', 'true', 'yes', 'on'
}
OUTPUT_DIR = Path(
    os.environ.get('FACE_DIAGNOSTIC_OUTPUT_DIR')
    or PROJECT_ROOT / 'deeplearning/training/runs/face_detection'
)
if not DETECTOR_PATH.is_file():
    raise FileNotFoundError(DETECTOR_PATH)
if FACE_RTSP_URL and not FACE_RTSP_URL.startswith('rtsp://'):
    raise ValueError('FACE_RTSP_URL은 rtsp:// 로 시작해야 합니다.')


In [2]:
import cv2
import numpy as np
import torch

TORCH_DLL_DIR = Path(torch.__file__).resolve().parent / 'lib'
if os.name == 'nt':
    cudnn_dll = TORCH_DLL_DIR / 'cudnn64_9.dll'
    if not cudnn_dll.is_file():
        raise FileNotFoundError(cudnn_dll)
    os.environ['PATH'] = f"{TORCH_DLL_DIR}{os.pathsep}{os.environ.get('PATH', '')}"
    _torch_dll_handle = os.add_dll_directory(str(TORCH_DLL_DIR))
    _cudnn_handle = ctypes.WinDLL(str(cudnn_dll))

from insightface.model_zoo import get_model
from deeplearning.scrfd_detection import (
    FaceCandidateStatus,
    ScrfdBoxTracker,
    ScrfdCandidateDetector,
)

model = get_model(
    str(DETECTOR_PATH),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
model.prepare(
    ctx_id=0,
    input_size=(INPUT_SIZE, INPUT_SIZE),
    det_thresh=CANDIDATE_THRESHOLD,
)
providers = model.session.get_providers()
if providers[0] != 'CUDAExecutionProvider':
    raise RuntimeError(f'SCRFD가 CUDA에서 실행되지 않습니다: {providers}')
model.detect(np.zeros((INPUT_SIZE, INPUT_SIZE, 3), dtype=np.uint8), max_num=0)
detector = ScrfdCandidateDetector(
    model,
    candidate_threshold=CANDIDATE_THRESHOLD,
    face_threshold=FACE_THRESHOLD,
)
tracker = ScrfdBoxTracker(stale_cycles=TRACK_STALE_CYCLES)
print('SCRFD CUDA warm-up 성공:', providers)


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'enable_cudnn': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
SCRFD CUDA warm-up 성공: ['CUDAExecutionProvider', 'CPUExecutionProvider']


## 실시간 CCTV 얼굴 후보 검출

초록 `FACE`는 확실한 얼굴 후보, 노랑 `REVIEW`는 confidence가 낮아 사람이 확인해야 하는 후보입니다. `q`로 종료하면 얼굴 이미지 없이 성능 집계 JSON만 저장합니다.

In [3]:
def open_camera() -> tuple[cv2.VideoCapture, str]:
    if FACE_RTSP_URL:
        camera = cv2.VideoCapture(FACE_RTSP_URL)
        # 추론이 수신 FPS보다 느릴 때 오래된 RTSP 프레임이 쌓이지 않게 한다.
        camera.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        source_label = 'RTSP CCTV'
    else:
        camera = cv2.VideoCapture(
            CAMERA_INDEX,
            cv2.CAP_DSHOW if os.name == 'nt' else cv2.CAP_ANY,
        )
        source_label = f'local camera {CAMERA_INDEX}'
    if not camera.isOpened():
        camera.release()
        raise RuntimeError(f'{source_label} 입력을 열지 못했습니다.')
    return camera, source_label

def run_scrfd_demo() -> None:
    camera, source_label = open_camera()
    window_name = 'SCRFD face detection v10'
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO)
    if DISPLAY_FULLSCREEN:
        cv2.setWindowProperty(
            window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN
        )

    frame_index = 0
    tracked = ()
    frame_times: list[float] = []
    metrics = {
        'detection_cycles': 0,
        'candidate_total': 0,
        'face_total': 0,
        'review_total': 0,
        'confidence_total': 0.0,
        'face_size_total_pixels': 0,
        'detector_time_total_ms': 0.0,
    }
    source_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
    source_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))

    try:
        while True:
            frame_started = time.perf_counter()
            ok, frame = camera.read()
            if not ok:
                continue
            if source_width <= 0 or source_height <= 0:
                source_height, source_width = frame.shape[:2]

            frame_index += 1
            tile_due = frame_index == 1 or frame_index % TILE_INTERVAL == 0
            full_due = frame_index == 1 or frame_index % FULL_INTERVAL == 0
            mode = 'hold'
            detector_ms = 0.0
            if tile_due or full_due:
                detection_started = time.perf_counter()
                if tile_due:
                    detections = detector.detect_tiled(
                        frame,
                        rows=TILE_ROWS,
                        columns=TILE_COLUMNS,
                        overlap=TILE_OVERLAP,
                        include_full_frame=True,
                    )
                    mode = 'full+tiles'
                else:
                    detections = detector.detect(frame)
                    mode = 'full'
                detector_ms = (time.perf_counter() - detection_started) * 1000.0
                tracked = tracker.update(detections)
                metrics['detection_cycles'] += 1
                metrics['candidate_total'] += len(detections)
                metrics['detector_time_total_ms'] += detector_ms
                for detection in detections:
                    metrics['confidence_total'] += detection.confidence
                    left, top, right, bottom = detection.bbox
                    metrics['face_size_total_pixels'] += min(right - left, bottom - top)
                    key = (
                        'face_total'
                        if detection.status is FaceCandidateStatus.FACE
                        else 'review_total'
                    )
                    metrics[key] += 1

            for item in tracked:
                is_face = item.status is FaceCandidateStatus.FACE
                color = (0, 200, 0) if is_face else (0, 210, 255)
                label = 'FACE' if is_face else 'REVIEW'
                if item.missed_cycles:
                    label += f' HOLD{item.missed_cycles}'
                left, top, right, bottom = item.bbox
                cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
                cv2.putText(
                    frame,
                    f'T{item.track_id} {label} c={item.confidence:.2f}',
                    (left, max(25, top - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    color,
                    2,
                )

            elapsed = time.perf_counter() - frame_started
            frame_times.append(elapsed)
            if len(frame_times) > 30:
                frame_times.pop(0)
            fps = len(frame_times) / max(sum(frame_times), 1e-6)
            overlay = (
                f'{source_label} {source_width}x{source_height} | FPS {fps:.1f} | '
                f'{mode} {detector_ms:.1f}ms | boxes {len(tracked)} | '
                f'input {INPUT_SIZE}'
            )
            cv2.putText(
                frame, overlay, (15, 28), cv2.FONT_HERSHEY_SIMPLEX,
                0.65, (255, 255, 255), 2
            )
            cv2.imshow(window_name, frame)
            if cv2.waitKeyEx(1) & 0xFF == ord('q'):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()
        candidates = int(metrics['candidate_total'])
        cycles = int(metrics['detection_cycles'])
        summary = {
            'source_resolution': [source_width, source_height],
            'detector_input_size': INPUT_SIZE,
            'candidate_threshold': CANDIDATE_THRESHOLD,
            'face_threshold': FACE_THRESHOLD,
            **metrics,
            'average_candidates_per_cycle': candidates / cycles if cycles else None,
            'average_confidence': metrics['confidence_total'] / candidates if candidates else None,
            'average_face_size_pixels': metrics['face_size_total_pixels'] / candidates if candidates else None,
            'average_detector_time_ms': metrics['detector_time_total_ms'] / cycles if cycles else None,
        }
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_path = OUTPUT_DIR / f"scrfd-v10-{time.strftime('%Y%m%d-%H%M%S')}.json"
        output_path.write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        print('SCRFD v10 진단 결과 저장:', output_path)

run_scrfd_demo()


SCRFD v10 진단 결과 저장: C:\smart_office_monitoring\deeplearning\training\runs\face_detection\scrfd-v10-20260820-144109.json
